# 01. EDA — 스마트 창고 출고 지연 예측

**목표**: 데이터 구조 파악, 결측치 분석, 타겟 분포, 주요 피처 관계 탐색

## 1. 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = '../open/'

In [ ]:
train = pd.read_csv(DATA_DIR + 'train.csv')
test = pd.read_csv(DATA_DIR + 'test.csv')
layout = pd.read_csv(DATA_DIR + 'layout_info.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Layout: {layout.shape}')

In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

train.head()

## 2. 기본 통계 및 데이터 타입

In [ ]:
train.info()

In [ ]:
train.describe().T

## 3. 결측치 분석

In [ ]:
# Train 결측치 비율
train_null = train.isnull().mean().sort_values(ascending=False)
train_null_pct = train_null[train_null > 0]

print(f'결측치 있는 피처 수: {len(train_null_pct)} / {len(train.columns)}')
print(f'\n결측 비율 상위 20개:')
train_null_pct.head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Train 결측치 분포
train_null_pct.plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Train 결측치 비율')
axes[0].set_ylabel('비율')
axes[0].tick_params(axis='x', rotation=45)

# Test 결측치 분포
test_null = test.isnull().mean().sort_values(ascending=False)
test_null_pct = test_null[test_null > 0]
test_null_pct.plot.bar(ax=axes[1], color='coral')
axes[1].set_title('Test 결측치 비율')
axes[1].set_ylabel('비율')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'Test 결측치 있는 피처 수: {len(test_null_pct)}')

In [ ]:
# Train vs Test 결측 비율 비교
null_compare = pd.DataFrame({
    'train_null_pct': train.isnull().mean(),
    'test_null_pct': test.drop(columns=[TARGET], errors='ignore').isnull().mean()
}).sort_values('train_null_pct', ascending=False)

null_compare[null_compare.max(axis=1) > 0]

## 4. 타겟 변수 분석

In [ ]:
print(train[TARGET].describe())
print(f'\n왜도(skewness): {train[TARGET].skew():.4f}')
print(f'첨도(kurtosis): {train[TARGET].kurtosis():.4f}')
print(f'0인 비율: {(train[TARGET] == 0).mean():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 히스토그램
train[TARGET].hist(bins=100, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('타겟 분포')
axes[0].set_xlabel(TARGET)

# 로그 변환 히스토그램 (0 제외)
non_zero = train[train[TARGET] > 0][TARGET]
np.log1p(non_zero).hist(bins=100, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('타겟 log1p 분포 (0 제외)')
axes[1].set_xlabel(f'log1p({TARGET})')

# 박스플롯
train[TARGET].plot.box(ax=axes[2])
axes[2].set_title('타겟 박스플롯')

plt.tight_layout()
plt.show()

## 5. 시나리오 구조 분석

In [ ]:
print(f"Train 시나리오 수: {train['scenario_id'].nunique()}")
print(f"Test 시나리오 수: {test['scenario_id'].nunique()}")
print(f"Train 창고(layout) 수: {train['layout_id'].nunique()}")
print(f"Test 창고(layout) 수: {test['layout_id'].nunique()}")

# 시나리오당 행 수 분포
scenario_counts = train.groupby('scenario_id').size()
print(f'\n시나리오당 행 수:\n{scenario_counts.describe()}')

In [ ]:
# Train/Test 시나리오 겹침 여부
train_scenarios = set(train['scenario_id'].unique())
test_scenarios = set(test['scenario_id'].unique())
overlap = train_scenarios & test_scenarios
print(f'Train/Test 시나리오 겹침: {len(overlap)}개')

# Train/Test 레이아웃 겹침
train_layouts = set(train['layout_id'].unique())
test_layouts = set(test['layout_id'].unique())
layout_overlap = train_layouts & test_layouts
print(f'Train/Test 레이아웃 겹침: {len(layout_overlap)} / Train {len(train_layouts)} / Test {len(test_layouts)}')

In [ ]:
# 시나리오 내 타겟 변화 패턴 (샘플 5개)
sample_scenarios = train['scenario_id'].unique()[:5]

fig, ax = plt.subplots(figsize=(14, 5))
for sc in sample_scenarios:
    sc_data = train[train['scenario_id'] == sc].reset_index(drop=True)
    ax.plot(sc_data.index, sc_data[TARGET], marker='o', markersize=3, label=sc)

ax.set_xlabel('타임슬롯 순서')
ax.set_ylabel(TARGET)
ax.set_title('시나리오별 타겟 시계열 패턴 (샘플 5개)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Layout 보조 테이블 분석

In [ ]:
layout.head(10)

In [ ]:
layout.describe().T

In [ ]:
print('layout_type 분포:')
print(layout['layout_type'].value_counts())

# 레이아웃 타입별 평균 지연
train_layout = train.merge(layout, on='layout_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_layout.groupby('layout_type')[TARGET].mean().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('레이아웃 타입별 평균 지연')
axes[0].set_ylabel('평균 지연(분)')

train_layout.boxplot(column=TARGET, by='layout_type', ax=axes[1])
axes[1].set_title('레이아웃 타입별 지연 분포')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Layout 수치 피처와 타겟 상관관계
layout_num_cols = layout.select_dtypes(include='number').columns.tolist()
layout_corr = train_layout[layout_num_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('Layout 피처 vs 타겟 상관계수:')
layout_corr

## 7. 피처-타겟 상관관계

In [ ]:
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
num_cols = train[feature_cols].select_dtypes(include='number').columns.tolist()

corr_target = train[num_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

print(f'타겟과 상관계수 상위 20개:')
corr_target.head(20)

In [ ]:
# 상관계수 상위/하위 바 차트
fig, ax = plt.subplots(figsize=(12, 8))
top_corr = pd.concat([corr_target.head(15), corr_target.tail(15)])
colors = ['steelblue' if v >= 0 else 'coral' for v in top_corr.values]
top_corr.plot.barh(ax=ax, color=colors)
ax.set_title('타겟과 상관계수 (상위/하위 15개)')
ax.set_xlabel('상관계수')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 상관계수 상위 피처 scatter plot
top_features = corr_target.head(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for i, feat in enumerate(top_features):
    ax = axes[i // 3][i % 3]
    ax.scatter(train[feat], train[TARGET], alpha=0.05, s=1, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel(TARGET)
    ax.set_title(f'{feat} (r={corr_target[feat]:.3f})')

plt.tight_layout()
plt.show()

## 8. 피처 그룹별 분석

In [ ]:
# 주요 피처 그룹 정의
feature_groups = {
    '주문': ['order_inflow_15m', 'unique_sku_15m', 'avg_items_per_order',
             'urgent_order_ratio', 'heavy_item_ratio', 'cold_chain_ratio', 'sku_concentration'],
    '로봇': ['robot_active', 'robot_idle', 'robot_charging', 'robot_utilization',
            'avg_trip_distance', 'task_reassign_15m'],
    '배터리': ['battery_mean', 'battery_std', 'low_battery_ratio',
             'charge_queue_length', 'avg_charge_wait'],
    '혼잡도': ['congestion_score', 'max_zone_density', 'blocked_path_15m',
             'near_collision_15m'],
    '장애': ['fault_count_15m', 'avg_recovery_time'],
    '패킹': ['replenishment_overlap', 'pack_utilization', 'manual_override_ratio'],
}

for group_name, cols in feature_groups.items():
    group_corr = corr_target.reindex(cols).dropna().sort_values(key=abs, ascending=False)
    print(f'\n[{group_name}] 타겟 상관계수:')
    for c, v in group_corr.items():
        print(f'  {c:30s} {v:+.4f}')

## 9. Train vs Test 분포 비교

In [ ]:
# 상위 상관 피처에 대해 Train/Test 분포 비교
compare_features = corr_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, feat in enumerate(compare_features):
    ax = axes[i // 3][i % 3]
    train[feat].hist(bins=50, ax=ax, alpha=0.6, label='Train', density=True, color='steelblue')
    test[feat].hist(bins=50, ax=ax, alpha=0.6, label='Test', density=True, color='coral')
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Train vs Test 분포 비교 (상위 상관 피처)', y=1.02)
plt.tight_layout()
plt.show()

## 10. 피처 간 상관관계 히트맵

In [ ]:
# 상위 상관 피처 20개 + 타겟
top20 = corr_target.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(16, 14))
corr_matrix = train[top20 + [TARGET]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('상위 상관 피처 간 상관관계 히트맵')
plt.tight_layout()
plt.show()

## 11. 시간대(shift_hour) 분석

In [ ]:
if 'shift_hour' in train.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    train.groupby('shift_hour')[TARGET].mean().plot(ax=axes[0], marker='o', color='steelblue')
    axes[0].set_title('시간대별 평균 지연')
    axes[0].set_ylabel('평균 지연(분)')
    axes[0].set_xlabel('shift_hour')
    
    train.groupby('shift_hour')[TARGET].count().plot.bar(ax=axes[1], color='steelblue')
    axes[1].set_title('시간대별 데이터 수')
    axes[1].set_ylabel('개수')
    
    plt.tight_layout()
    plt.show()

## 12. 요약 및 인사이트

**여기에 EDA 실행 후 발견된 인사이트를 정리**

- 타겟 분포 특성: (실행 후 작성)
- 핵심 피처: (실행 후 작성)
- 결측치 전략: (실행 후 작성)
- 시계열 활용 가능성: (실행 후 작성)
- layout 보조 데이터 유용성: (실행 후 작성)